# Notlari PDF olarak Drive'a yaz

Notlarin HTML kaynagi ve baski stili (`kbb/notlar/`) git deposunda durur.
Bu defter depoyu klonlar, her notu PDF'e cevirir ve Drive'a yazar:

- gun notlari -> `KBB_not_claude/`
- `soru-` ile baslayan soru notlari -> `KBB_not_claude/Soru/`

HTML'i degismemis notlari atlar, yani her calistirmada sadece yeni ve
guncellenmis notlar islenir.

Hucreleri sirayla calistir - duzenlemen gereken bir sey yok.

In [ ]:
# 1) Drive'i bagla
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2) Araclari kur
!pip install -q weasyprint
!apt-get -qq install -y fonts-liberation > /dev/null
print("kurulum tamam")


## Eski notlarin gorsellerini geri kazan (istege bagli)

Bir gun notu yeniden uretilirken eski PDF'in icindeki fotograflar kaybolmasin
diye bu hucre eski PDF'lerden resimleri cikarip
`KBB_not_claude/gorsel/<not-adi>/rNN.png` altina yazar. HTML'de bunlara
`<img src="gorsel/<not-adi>/r01.png">` ile basvurulur.

Sadece yeniden uretilecek notlar icin calistir; `NOTLAR` listesini duzenle.

In [ ]:
# Eski PDF'lerden gorsel cikar
#
# Yeniden uretilecek not adlarini yaz (uzantisiz). Bos birakirsan
# KBB_not_claude altindaki butun PDF'ler islenir - uzun surer.

NOTLAR = ["gun-15-septum-deviasyonu-septoplasti"]

!pip install -q pymupdf

import glob, os
import pymupdf

HEDEF = "/content/drive/MyDrive/PAÜ/KBB_not_claude"
if not os.path.isdir(HEDEF):
    HEDEF = glob.glob("/content/drive/MyDrive/**/KBB_not_claude", recursive=True)[0]

GORSEL = os.path.join(HEDEF, "gorsel")
os.makedirs(GORSEL, exist_ok=True)

adlar = NOTLAR or [os.path.splitext(os.path.basename(p))[0]
                   for p in glob.glob(os.path.join(HEDEF, "*.pdf"))]

for ad in adlar:
    kaynak = os.path.join(HEDEF, ad + ".pdf")
    if not os.path.exists(kaynak):
        print(f"  yok, atlandi: {ad}.pdf")
        continue
    klasor = os.path.join(GORSEL, ad)
    os.makedirs(klasor, exist_ok=True)

    belge = pymupdf.open(kaynak)
    sayac = 0
    gorulen = set()
    for sayfa in belge:
        for bilgi in sayfa.get_images(full=True):
            xref = bilgi[0]
            if xref in gorulen:
                continue
            gorulen.add(xref)
            pix = pymupdf.Pixmap(belge, xref)
            if pix.n - pix.alpha >= 4:          # CMYK -> RGB
                pix = pymupdf.Pixmap(pymupdf.csRGB, pix)
            # Cizgi/ikon boyutundaki artiklari atla
            if pix.width < 120 or pix.height < 120:
                continue
            sayac += 1
            pix.save(os.path.join(klasor, f"r{sayac:02d}.png"))
    print(f"  {ad}: {sayac} gorsel -> gorsel/{ad}/")

print("\nBitti. HTML'de: <img src=\"gorsel/<not-adi>/r01.png\">")

In [ ]:
# 3) Depoyu cek ve notlari PDF'e cevir
#
# Notlarin HTML kaynagi ve baski stili git deposunda tutulur; bu hucre
# depoyu klonlar, kbb/notlar/ altindaki her HTML'i PDF'e cevirir ve
# Drive'a yazar. Adi "soru-" ile baslayan notlar Soru/ alt klasorune,
# gun notlari dogrudan KBB_not_claude'a gider. HTML degismemisse atlar.

import glob, os, shutil, subprocess
from weasyprint import HTML

DEPO = "https://github.com/alpercil/Repository1"
YEREL = "/content/repo"

if os.path.exists(YEREL):
    shutil.rmtree(YEREL)
subprocess.run(["git", "clone", "--depth", "1", "-q", DEPO, YEREL], check=True)

# Hedef klasor: dogrulanmis yol, bulunamazsa ara
HEDEF = "/content/drive/MyDrive/PAÜ/KBB_not_claude"
if not os.path.isdir(HEDEF):
    adaylar = glob.glob("/content/drive/MyDrive/**/KBB_not_claude", recursive=True)
    if not adaylar:
        raise SystemExit("KBB_not_claude bulunamadi - HEDEF'i elle yaz")
    HEDEF = adaylar[0]

SORU = os.path.join(HEDEF, "Soru")
os.makedirs(SORU, exist_ok=True)
print("hedef klasor:", HEDEF)
print("soru klasoru:", SORU)

kaynak = os.path.join(YEREL, "kbb", "notlar")
for html in sorted(glob.glob(os.path.join(kaynak, "*.html"))):
    ad = os.path.splitext(os.path.basename(html))[0]
    # Soru notlari ayri klasore
    klasor = SORU if ad.startswith("soru-") else HEDEF
    cikti = os.path.join(klasor, ad + ".pdf")
    # HTML degismisse PDF'i yenile; degismemisse atla
    if os.path.exists(cikti) and os.path.getmtime(cikti) >= os.path.getmtime(html):
        print(f"  atlandi (guncel): {ad}.pdf")
        continue
    HTML(html, base_url=kaynak).write_pdf(cikti)
    nere = "Soru/" if klasor is SORU else ""
    print(f"  uretildi: {nere}{ad}.pdf  {os.path.getsize(cikti)/1e6:.1f} MB")

print("\nBitti.")